<a href="https://colab.research.google.com/github/sabithakrishnan/multimodal-image-analysis/blob/main/multimodalimage_recognition_linear_probe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install open-clip-torch datasets pillow torch torchvision matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00


In [ ]:

import os
from pathlib import Path
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

class ProductionDatasetBuilder:
    def __init__(self, dataset_repo: str = "blanchon/EuroSAT_RGB"):
        self.repo = dataset_repo

    def build_local_directory(self, output_dir: str = "satellite_dataset"):
        dest_path = Path(output_dir)

        # Guard clause: avoid re-running if data exists
        if dest_path.exists() and any(dest_path.iterdir()):
            print(f"📦 Workspace directory '{output_dir}' already exists. Transitioning steps...")
            return

        print(f"📡 Accessing Hugging Face native parquet tables for '{self.repo}'...")
        # Loads clean tabular shards; completely bypasses legacy python execution scripts
        dataset_split = load_dataset(self.repo, split="train")

        # Dynamically infer the land-use taxonomy from the Metadata Table schema
        label_names = dataset_split.features["label"].names
        print(f"🎯 Verified {len(label_names)} target categories for extraction.")

        print("\n💾 Unpacking and storing satellite tensors to local filesystems...")
        for idx, sample in enumerate(tqdm(dataset_split, desc="Processing Satellite Tiles")):
            pil_img = sample["image"]
            label_id = sample["label"]

            # Sanitize naming structures to eliminate runtime mapping mistakes
            class_name = label_names[label_id].replace(" ", "")

            # Map path structure: satellite_dataset/IndustrialBuildings/tile_x.jpg
            class_folder = dest_path / class_name
            class_folder.mkdir(parents=True, exist_ok=True)

            file_name = class_folder / f"tile_{idx}.jpg"

            if pil_img.mode != "RGB":
                pil_img = pil_img.convert("RGB")

            pil_img.save(file_name, "JPEG")

        print(f"\n✅ Step 2 Complete! File mapping successfully resolved inside: '{dest_path.resolve()}'")

if __name__ == "__main__":
    builder = ProductionDatasetBuilder()
    builder.build_local_directory()


📡 Accessing Hugging Face native parquet tables for 'blanchon/EuroSAT_RGB'...


README.md:   0%|          | 0.00/3.38k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  105MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 34.8MB            

...

Generating test split:   0%|          | 0/5400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5400 [00:00<?, ? examples/s]

🎯 Verified 10 target categories for extraction.

💾 Unpacking and storing satellite tensors to local filesystems...


Processing Satellite Tiles: 100%|██████████| 16200/16200 [00:14<00:00, 1090.31it/s]


✅ Step 2 Complete! File mapping successfully resolved inside: '/content/satellite_dataset'


In [ ]:
#without prompt using linear probe
import torch
import torch.nn as nn
import torch.optim as optim
import open_clip
from torch.utils.data import DataLoader, TensorDataset, random_split, Dataset
from tqdm import tqdm
import os
from PIL import Image
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix
import numpy as np

def train_linear_probe():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_directory = "satellite_dataset"

    #  Load CLIP for only Feature Extraction
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k', device=device)
    model.eval()

    #  Extract Data
    dataset = ImageFolder(root=data_directory, transform=preprocess)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False)

    all_embeddings = []
    all_labels = []

    # Pass all images through the frozen vision tower to cache their vectors
    print("💾 Extracting visual feature maps (this runs once)...")
    with torch.no_grad():
        for images, labels in tqdm(dataloader):
            images = images.to(device)
            image_features = model.encode_image(images)
            image_features /= image_features.norm(dim=-1, keepdim=True) # L2 Normalize

            all_embeddings.append(image_features.cpu())
            all_labels.append(labels)

    # Concatenate into static training matrices
    X = torch.cat(all_embeddings, dim=0)
    Y = torch.cat(all_labels, dim=0)

    # Create a clean Train/Val Split for your new vectors
    feature_dataset = TensorDataset(X, Y)
    train_size = int(0.8 * len(feature_dataset))
    val_size = len(feature_dataset) - train_size
    train_set, val_set = random_split(feature_dataset, [train_size, val_size])

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=128, shuffle=False)

    # Define the Linear Classifier Top Head (No Prompts Used)
    num_classes = len(dataset.classes)
    classifier = nn.Linear(512, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-2)

    # Fast Training Loop (Takes seconds/minutes)
    print("\n🚀 Training linear probe on top of image embeddings...")
    for epoch in range(10): # 10 epochs is plenty for a linear layer
        classifier.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = classifier(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation Check
        classifier.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                preds = classifier(batch_x).argmax(dim=-1)
                correct += (preds == batch_y).sum().item()
                total += batch_y.size(0)

        print(f"Epoch {epoch+1:02d} | Loss: {train_loss/len(train_loader):.4f} | Val Accuracy: {(correct/total)*100:.2f}%")
        # final predictions and final validation predictions
    classifier.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            preds = classifier(batch_x).argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_trues.extend(batch_y.cpu().numpy())

    # Generate Confusion Matrix
    cm = confusion_matrix(all_trues, all_preds)
    total_samples = np.sum(cm)
    tp = np.sum(np.diag(cm))
    fp = np.sum(cm) - tp
    fn = np.sum(cm) - tp
    num_classes = len(dataset.classes)
    tn = (num_classes * total_samples) - (tp + fp + fn)

    # Compute Overall Metrics (Adding 1e-7 prevents division-by-zero)
    accuracy  = (tp+tn) / ( tp + tn + fp + fn + 1e-7)
    precision = tp / (tp + fp + 1e-7)
    recall    = tp / (tp + fn + 1e-7)
    f1        = 2 * (precision * recall) / (precision + recall + 1e-7)

    print("\n🌍 ---  Performance Metrics ---")

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")


if __name__ == "__main__":
    train_linear_probe()


💾 Extracting visual feature maps (this runs once)...


100%|██████████| 254/254 [01:21<00:00,  3.11it/s]



🚀 Training linear probe on top of image embeddings...
Epoch 01 | Loss: 2.0813 | Val Accuracy: 60.00%
Epoch 02 | Loss: 1.7362 | Val Accuracy: 73.12%
Epoch 03 | Loss: 1.4827 | Val Accuracy: 77.35%
Epoch 04 | Loss: 1.2933 | Val Accuracy: 79.07%
Epoch 05 | Loss: 1.1495 | Val Accuracy: 80.62%
Epoch 06 | Loss: 1.0374 | Val Accuracy: 81.42%
Epoch 07 | Loss: 0.9482 | Val Accuracy: 82.07%
Epoch 08 | Loss: 0.8761 | Val Accuracy: 82.72%
Epoch 09 | Loss: 0.8163 | Val Accuracy: 83.12%
Epoch 10 | Loss: 0.7679 | Val Accuracy: 83.92%

🌍 ---  Performance Metrics ---
Accuracy:  0.9678
Precision: 0.8392
Recall:    0.8392
F1-Score:  0.8392
